In [3]:
!pip install torch pandas torchvision pillow scikit-learn

  Using cached pandas-3.0.5-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached torchvision-0.28.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (5.6 kB)
  Using cached pillow-12.3.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.1 kB)
  Using cached scikit_learn-1.9.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached numpy-2.4.6-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached scipy-1.17.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached narwhals-2.23.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached pandas-3.0.5-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (11.3 MB)
Using cached torchvision-0.28.0-cp311-cp311-manylinux_2_28_x86_64.whl (7.7 MB)
Using

In [4]:
from pathlib import Path
import os
import sys
import subprocess
import json
import time
import shutil
import pandas as pd
import numpy as np
from PIL import Image, ImageFile
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")
FAKE_TEST_ROOT = PROJECT_ROOT / "df40" / "Fake_Test"

OOD_OUT_DIR = PROJECT_ROOT / "outputs_df40_ood_fake_test_all_models"
OOD_OUT_DIR.mkdir(parents=True, exist_ok=True)

# Existing trained model folders
GRFN_SCRIPT = PROJECT_ROOT / "grfn_csf_grfn_only_v6.py"
GRFN_OUT = PROJECT_ROOT / "outputs_grfn_df40_indomain_auto_balance"

FREQ_SCRIPT = PROJECT_ROOT / "freqdebias_ffpp.py"
FREQ_OUT = PROJECT_ROOT / "outputs_freqdebias_df40_indomain"

GEND_SCRIPT = PROJECT_ROOT / "gend_reimpl_ffpp.py"
GEND_OUT = PROJECT_ROOT / "outputs_gend_reimpl_df40_indomain"

DFDHR_REPO = PROJECT_ROOT / "DFD-HR"
DFDHR_RUNNER = DFDHR_REPO / "run_dfd_hr_df40.py"
DFDHR_CKPT = DFDHR_REPO / "ckpt_best.pth"

print("Project root:", PROJECT_ROOT)
print("Fake_Test root exists:", FAKE_TEST_ROOT.exists(), FAKE_TEST_ROOT)
print("Output dir:", OOD_OUT_DIR)

Project root: /home/jovyan/Deepfake_Low Resource_Low Resolution
Fake_Test root exists: True /home/jovyan/Deepfake_Low Resource_Low Resolution/df40/Fake_Test
Output dir: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40_ood_fake_test_all_models


/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

VERIFY_IMAGES = True  # set False if you already verified this folder

def infer_method_from_path(path, root):
    rel = Path(path).relative_to(root)
    if len(rel.parts) >= 2:
        return rel.parts[0]
    return "unknown_method"

def crawl_fake_test(fake_root, verify_images=True):
    fake_root = Path(fake_root)
    if not fake_root.exists():
        raise FileNotFoundError(f"Fake_Test folder not found: {fake_root}")

    paths = []
    for p in fake_root.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            paths.append(p)

    paths = sorted(paths)
    print("Found image candidates:", len(paths))

    rows = []
    bad_rows = []

    for i, p in enumerate(tqdm(paths, desc="Building Fake_Test manifest")):
        method = infer_method_from_path(p, fake_root)

        ok = True
        reason = ""
        if verify_images:
            try:
                if p.stat().st_size == 0:
                    ok = False
                    reason = "empty_file"
                else:
                    with Image.open(p) as img:
                        img.verify()
                    with Image.open(p) as img:
                        img.convert("RGB")
            except Exception as e:
                ok = False
                reason = f"{type(e).__name__}: {e}"

        if not ok:
            bad_rows.append({
                "frame_path": str(p),
                "method": method,
                "reason": reason,
                "file_size_bytes": p.stat().st_size if p.exists() else None,
            })
            continue

        # Use one unique group/video id per image so no script accidentally samples away images.
        rel_no_ext = str(p.relative_to(fake_root).with_suffix(""))
        video_id = f"{method}|{rel_no_ext}|{i}"

        rows.append({
            "frame_path": str(p),
            "image_path": str(p),
            "path": str(p),
            "label": 1,
            "label_name": "fake",
            "method": method,
            "domain": "Fake_Test",
            "source_layout": "df40_fake_test",
            "top_level_folder": method,
            "extension": p.suffix.lower(),
            "group_id": video_id,
            "video_id": video_id,
            "sample_unit_id": f"Fake_Test|1|{video_id}",
            "split": "test",
        })

    df_test = pd.DataFrame(rows)
    bad_df = pd.DataFrame(bad_rows)

    if len(df_test) == 0:
        raise RuntimeError("No valid Fake_Test images found.")

    # Some existing scripts require non-empty train/val splits even in eval-only mode.
    # These rows are not used for final scoring, but prevent loader failures.
    dummy_n = min(16, len(df_test))
    dummy_train = df_test.head(dummy_n).copy()
    dummy_train["split"] = "train"

    dummy_val = df_test.tail(dummy_n).copy()
    dummy_val["split"] = "val"

    df_all = pd.concat([dummy_train, dummy_val, df_test], ignore_index=True)

    manifest_path = OOD_OUT_DIR / "df40_fake_test_manifest_for_eval.csv"
    test_only_path = OOD_OUT_DIR / "df40_fake_test_manifest_test_only.csv"
    bad_path = OOD_OUT_DIR / "df40_fake_test_bad_images.csv"

    df_all.to_csv(manifest_path, index=False)
    df_test.to_csv(test_only_path, index=False)
    bad_df.to_csv(bad_path, index=False)

    print("Valid test images:", len(df_test))
    print("Bad images:", len(bad_df))
    print("Methods:", df_test["method"].nunique())
    print("Saved manifest:", manifest_path)

    display(df_test["method"].value_counts().head(20).to_frame("num_images"))

    return df_all, df_test, manifest_path, test_only_path

fake_all_df, fake_test_df, FAKE_MANIFEST, FAKE_TEST_ONLY_MANIFEST = crawl_fake_test(
    FAKE_TEST_ROOT,
    verify_images=VERIFY_IMAGES,
)

Found image candidates: 175667


Building Fake_Test manifest:   0%|          | 627/175667 [00:35<2:47:13, 17.45it/s]


KeyboardInterrupt: 

In [5]:
def run_cmd(cmd, cwd=PROJECT_ROOT):
    print("\n" + "=" * 100)
    print("Running:")
    print(" ".join([str(c) for c in cmd]))
    print("=" * 100)
    t0 = time.time()
    result = subprocess.run(
        [str(c) for c in cmd],
        cwd=str(cwd),
        text=True,
    )
    elapsed = time.time() - t0
    print(f"Finished in {elapsed/60:.2f} minutes with return code {result.returncode}")
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with return code {result.returncode}")
    return elapsed

def file_exists(path):
    return Path(path).exists()

def find_first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

def read_grfn_eval_settings(grfn_out):
    """
    Use the in-domain threshold and score direction, not a threshold tuned on fake-only OOD data.
    """
    summary_path = Path(grfn_out) / "experiment_summary.csv"
    threshold = 0.5
    score_direction = "normal"

    if summary_path.exists():
        s = pd.read_csv(summary_path)
        row = s.iloc[-1]

        if "Decision Threshold" in row and pd.notna(row["Decision Threshold"]):
            threshold = float(row["Decision Threshold"])

        if "Score Inverted For Eval" in row:
            val = str(row["Score Inverted For Eval"]).strip().lower()
            if val in ["true", "1", "yes"]:
                score_direction = "invert"

    return threshold, score_direction

def script_supports(script_path, flag):
    script_path = Path(script_path)
    if not script_path.exists():
        return False
    try:
        r = subprocess.run(
            [sys.executable, str(script_path), "--help"],
            cwd=str(PROJECT_ROOT),
            capture_output=True,
            text=True,
            timeout=60,
        )
        text = (r.stdout or "") + "\n" + (r.stderr or "")
        return flag in text
    except Exception:
        return False

def load_prediction_csv(model_name, pred_path, threshold=0.5):
    pred_path = Path(pred_path)
    if not pred_path.exists():
        print(f"Missing predictions for {model_name}: {pred_path}")
        return None

    pred = pd.read_csv(pred_path)

    path_col = None
    for c in ["image_path", "frame_path", "path"]:
        if c in pred.columns:
            path_col = c
            break

    if path_col is None:
        raise ValueError(f"{model_name}: could not find image path column in {pred.columns.tolist()}")

    if "fake_prob" not in pred.columns:
        raise ValueError(f"{model_name}: prediction file missing fake_prob column.")

    out = pred.copy()
    out["image_path"] = out[path_col].astype(str)
    out["fake_prob"] = out["fake_prob"].astype(float)
    out["model"] = model_name
    out["threshold"] = float(threshold)
    out["pred_fake"] = (out["fake_prob"] >= float(threshold)).astype(int)

    method_map = fake_test_df.set_index("image_path")["method"].to_dict()
    out["method"] = out["image_path"].map(method_map).fillna("unknown_method")

    return out

def summarize_fake_only_predictions(pred_df):
    """
    Fake-only OOD metrics.
    Since every image is fake, Recall(1) = predicted fake rate.
    """
    overall = (
        pred_df.groupby("model")
        .agg(
            num_images=("fake_prob", "size"),
            mean_fake_prob=("fake_prob", "mean"),
            median_fake_prob=("fake_prob", "median"),
            fake_detection_rate_recall1=("pred_fake", "mean"),
            missed_fake_rate=("pred_fake", lambda x: 1.0 - float(np.mean(x))),
            threshold=("threshold", "first"),
        )
        .reset_index()
    )

    by_method = (
        pred_df.groupby(["model", "method"])
        .agg(
            num_images=("fake_prob", "size"),
            mean_fake_prob=("fake_prob", "mean"),
            median_fake_prob=("fake_prob", "median"),
            fake_detection_rate_recall1=("pred_fake", "mean"),
            missed_fake_rate=("pred_fake", lambda x: 1.0 - float(np.mean(x))),
            threshold=("threshold", "first"),
        )
        .reset_index()
        .sort_values(["model", "fake_detection_rate_recall1", "num_images"], ascending=[True, True, False])
    )

    pivot = by_method.pivot_table(
        index="method",
        columns="model",
        values="fake_detection_rate_recall1",
        aggfunc="first",
    ).reset_index()

    return overall, by_method, pivot

In [ ]:
GRFN_OOD_OUT = OOD_OUT_DIR / "outputs_grfn_df40_indomain_auto_balance"

grfn_ckpt = find_first_existing([
    GRFN_OUT / "best_checkpoint.pt",
    GRFN_OUT / "latest_checkpoint.pt",
])

print("GRFN script:", GRFN_SCRIPT, GRFN_SCRIPT.exists())
print("GRFN checkpoint:", grfn_ckpt)

if GRFN_SCRIPT.exists() and grfn_ckpt is not None:
    grfn_threshold, grfn_score_direction = read_grfn_eval_settings(GRFN_OUT)
    print("GRFN threshold:", grfn_threshold)
    print("GRFN score direction:", grfn_score_direction)

    cmd = [
        sys.executable, GRFN_SCRIPT,
        "--metadata_csv", FAKE_MANIFEST,
        "--out_dir", GRFN_OOD_OUT,
        "--dataset_name", "DF40_Fake_Test",
        "--resume", grfn_ckpt,
        "--eval_only",
        "--decision_threshold", str(grfn_threshold),
        "--score_direction", grfn_score_direction,
        "--frames_per_video", "1",
        "--batch_size", "16",
        "--image_size", "224",
        "--num_workers", "0",
        "--balance_strategy", "none",
        "--amp",
    ]

    run_cmd(cmd, cwd=PROJECT_ROOT)
else:
    print("Skipping GRFN because script/checkpoint was not found.")

In [10]:
from pathlib import Path
import pandas as pd

# Use the test-only manifest created earlier
src = Path("outputs_df40_ood_fake_test_all_models/df40_fake_test_manifest_test_only.csv")

# If your notebook variable exists, this is also fine:
# src = Path(FAKE_TEST_ONLY_MANIFEST)

df = pd.read_csv(src)

# Keep only columns FreqDebias expects, plus label_name is harmless.
freq_df = pd.DataFrame({
    "frame_path": df["frame_path"].astype(str),
    "split": "test",
    "label": 1,
    "label_name": "fake",
    "video_id": df["video_id"].astype(str),
})

# Add dummy train/val rows so the old script does not crash on empty splits.
dummy_n = min(16, len(freq_df))

dummy_train = freq_df.head(dummy_n).copy()
dummy_train["split"] = "train"

dummy_val = freq_df.tail(dummy_n).copy()
dummy_val["split"] = "val"

freq_all = pd.concat([dummy_train, dummy_val, freq_df], ignore_index=True)

FREQ_FAKE_MANIFEST = Path("outputs_df40_ood_fake_test_all_models/df40_fake_test_manifest_freqdebias.csv")
freq_all.to_csv(FREQ_FAKE_MANIFEST, index=False)

print("Saved:", FREQ_FAKE_MANIFEST)
print(freq_all.columns.tolist())
print(freq_all["split"].value_counts())
display(freq_all.head())

Saved: outputs_df40_ood_fake_test_all_models/df40_fake_test_manifest_freqdebias.csv
['frame_path', 'split', 'label', 'label_name', 'video_id']
split
test     175667
train        16
val          16
Name: count, dtype: int64


,frame_path,split,label,label_name,video_id
0,/home/jovyan/Deepfake_Low Resource_Low Resolut...,train,1,fake,CollabDiff|CollabDiff/fake/0/000000_x_0|0
1,/home/jovyan/Deepfake_Low Resource_Low Resolut...,train,1,fake,CollabDiff|CollabDiff/fake/1/000000_x_0|1
2,/home/jovyan/Deepfake_Low Resource_Low Resolut...,train,1,fake,CollabDiff|CollabDiff/fake/10/000000_x_0|2
3,/home/jovyan/Deepfake_Low Resource_Low Resolut...,train,1,fake,CollabDiff|CollabDiff/fake/100/000000_x_0|3
4,/home/jovyan/Deepfake_Low Resource_Low Resolut...,train,1,fake,CollabDiff|CollabDiff/fake/101/000000_x_0|4


In [12]:
FREQ_OOD_OUT = OOD_OUT_DIR / "freqdebias"

freq_ckpt = find_first_existing([
    FREQ_OUT / "best_checkpoint.pt",
    FREQ_OUT / "freqdebias_latest.pt",
    FREQ_OUT / "latest_checkpoint.pt",
])

print("FreqDebias script:", FREQ_SCRIPT, FREQ_SCRIPT.exists())
print("FreqDebias checkpoint:", freq_ckpt)
print("FreqDebias manifest:", FREQ_FAKE_MANIFEST)

if FREQ_SCRIPT.exists() and freq_ckpt is not None:
    cmd = [
        sys.executable, FREQ_SCRIPT,
        "--metadata_csv", FREQ_FAKE_MANIFEST,
        "--out_dir", FREQ_OOD_OUT,
        "--resume", freq_ckpt,
        "--skip_pretrain",
        "--pretrain_epochs", "0",
        "--freqdebias_epochs", "0",
        "--frames_per_video", "1",
        "--batch_size", "16",
        "--image_size", "256",
        "--num_workers", "0",
    ]

    run_cmd(cmd, cwd=PROJECT_ROOT)
else:
    print("Skipping FreqDebias because script/checkpoint was not found.")

FreqDebias script: /home/jovyan/Deepfake_Low Resource_Low Resolution/freqdebias_ffpp.py True
FreqDebias checkpoint: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_freqdebias_df40_indomain/freqdebias_latest.pt
FreqDebias manifest: outputs_df40_ood_fake_test_all_models/df40_fake_test_manifest_freqdebias.csv

Running:
/opt/conda/bin/python /home/jovyan/Deepfake_Low Resource_Low Resolution/freqdebias_ffpp.py --metadata_csv outputs_df40_ood_fake_test_all_models/df40_fake_test_manifest_freqdebias.csv --out_dir /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40_ood_fake_test_all_models/freqdebias --resume /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_freqdebias_df40_indomain/freqdebias_latest.pt --skip_pretrain --pretrain_epochs 0 --freqdebias_epochs 0 --frames_per_video 1 --batch_size 16 --image_size 256 --num_workers 0

FreqDebias FF++ regular experiment
Device: cuda
Output dir: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40_ood_fake_test_a

Sampling frames/video: 100%|██████████| 175667/175667 [28:51<00:00, 101.45video/s]


Sampled frames=175667, videos=175667, real=0, fake=175667

Building ResNet-34 FreqDebias model
Loading checkpoint: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_freqdebias_df40_indomain/freqdebias_latest.pt

FreqDebias training

Final test evaluation


Predict test: 100%|██████████| 10980/10980 [42:50<00:00,  4.27it/s] 



Test metrics:
Level  Accuracy  AUPRC  AUCROC  EER (Equal Error Rate)  Precision (0)  Recall (0)  F1 (0)  Precision (1)  Recall (1)   F1 (1)  Uncertain Rate (%)
Frame  0.988706    NaN     NaN                     NaN            0.0         0.0     0.0            1.0    0.988706 0.994321            0.015939
Video  0.988706    NaN     NaN                     NaN            0.0         0.0     0.0            1.0    0.988706 0.994321            0.015939

Inference:
 Frames  Videos  Total Seconds  Time Per Frame (ms)       FPS Device
 175667  175667     2570.47935            14.632682 68.340172   cuda

Saved outputs to: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40_ood_fake_test_all_models/freqdebias
Finished in 72.02 minutes with return code 0


In [6]:
from pathlib import Path
import pandas as pd

# This is the safe manifest already created for FreqDebias.
# It also works for GenD because GenD expects frame_path, split, label, video_id.
OOD_SAFE_MANIFEST = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution/"
    "outputs_df40_ood_fake_test_all_models/"
    "df40_fake_test_manifest_freqdebias.csv"
)

print("OOD_SAFE_MANIFEST:", OOD_SAFE_MANIFEST)
print("Exists:", OOD_SAFE_MANIFEST.exists())

df_check = pd.read_csv(OOD_SAFE_MANIFEST)
print("Columns:", df_check.columns.tolist())
print(df_check["split"].value_counts())
display(df_check.head())

OOD_SAFE_MANIFEST: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40_ood_fake_test_all_models/df40_fake_test_manifest_freqdebias.csv
Exists: True
Columns: ['frame_path', 'split', 'label', 'label_name', 'video_id']
split
test     175667
train        16
val          16
Name: count, dtype: int64


,frame_path,split,label,label_name,video_id
0,/home/jovyan/Deepfake_Low Resource_Low Resolut...,train,1,fake,CollabDiff|CollabDiff/fake/0/000000_x_0|0
1,/home/jovyan/Deepfake_Low Resource_Low Resolut...,train,1,fake,CollabDiff|CollabDiff/fake/1/000000_x_0|1
2,/home/jovyan/Deepfake_Low Resource_Low Resolut...,train,1,fake,CollabDiff|CollabDiff/fake/10/000000_x_0|2
3,/home/jovyan/Deepfake_Low Resource_Low Resolut...,train,1,fake,CollabDiff|CollabDiff/fake/100/000000_x_0|3
4,/home/jovyan/Deepfake_Low Resource_Low Resolut...,train,1,fake,CollabDiff|CollabDiff/fake/101/000000_x_0|4


In [8]:
!pip install transformers

  Using cached transformers-5.14.1-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-1.25.1-py3-none-any.whl.metadata (16 kB)
  Using cached regex-2026.7.19-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached typer-0.27.0-py3-none-any.whl.metadata (15 kB)
  Using cached safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.2 kB)
  Using cached click-8.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached hf_xet-1.5.2-cp38-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.9 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached annotated_doc-0.0.5-py3-none-any.whl.metadata (6.5 kB)
  Using cached markdown_it_py-4.2.0-py3-none-any.whl.metadata (7.4 kB)
  Using cach

In [11]:
GEND_OOD_OUT = OOD_OUT_DIR / "gend_ood_fake_test"

cmd = [
    sys.executable, "-u", str(GEND_SCRIPT),
    "--metadata_csv", str(OOD_SAFE_MANIFEST),
    "--out_dir", str(GEND_OOD_OUT),
    "--checkpoint", str(gend_ckpt),
    "--eval_only",
    "--split", "test",
    "--frames_per_video", "1",
    "--batch_size", "16",
    "--image_size", "224",
    "--num_workers", "0",
    "--amp",
]

log_path = GEND_OOD_OUT / "gend_full_live.log"
GEND_OOD_OUT.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

print("Running full GenD OOD test.")
print("Log:", log_path)

with open(log_path, "w") as log:
    process = subprocess.Popen(
        cmd,
        cwd=str(PROJECT_ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    for line in process.stdout:
        print(line, end="")
        log.write(line)
        log.flush()

    return_code = process.wait()

print("Return code:", return_code)
if return_code != 0:
    raise RuntimeError(f"GenD full run failed. Check log: {log_path}")

Running full GenD OOD test.
Log: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40_ood_fake_test_all_models/gend_ood_fake_test/gend_full_live.log

GenD-style FF++ reimplementation
Device: cuda
Backbone: openai/clip-vit-large-patch14
Output dir: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40_ood_fake_test_all_models/gend_ood_fake_test
LN tuning: True
Uniformity/alignment losses: True

FF++ metadata loaded
CSV: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40_ood_fake_test_all_models/df40_fake_test_manifest_freqdebias.csv
Total usable frames: 175699
train: frames=16, videos=16, real=0, fake=16
val: frames=16, videos=16, real=0, fake=16
test: frames=175667, videos=175667, real=0, fake=175667

Sampling frames

Sample train: 100%|██████████| 16/16 [00:00<00:00, 2491.88video/s]
Sampled frames=16, videos=16, real=0, fake=16

Sample val: 100%|██████████| 16/16 [00:00<00:00, 2754.54video/s]
Sampled frames=16, videos=16, real=0, fake=16

Sample test: 10

In [14]:
DFDHR_OOD_OUT = OOD_OUT_DIR / "outputs_dfd_hr_df40_indomain_test"

print("DFD-HR runner:", DFDHR_RUNNER, DFDHR_RUNNER.exists())
print("DFD-HR checkpoint:", DFDHR_CKPT, DFDHR_CKPT.exists())

if DFDHR_RUNNER.exists() and DFDHR_CKPT.exists():
    cmd = [
        sys.executable, DFDHR_RUNNER,
        "--repo_root", DFDHR_REPO,
        "--metadata_csv", FAKE_MANIFEST,
        "--ckpt_path", DFDHR_CKPT,
        "--detector_yaml", DFDHR_REPO / "training" / "config" / "detector" / "dfd_hr.yaml",
        "--test_config", DFDHR_REPO / "training" / "config" / "test_config.yaml",
        "--out_dir", DFDHR_OOD_OUT,
        "--split", "test",
        "--image_size", "448",
        "--frames_per_video", "1",
        "--batch_size", "1",
        "--allow_non_strict",
    ]

    run_cmd(cmd, cwd=PROJECT_ROOT)
else:
    print("Skipping DFD-HR because runner/checkpoint was not found.")

DFD-HR runner: /home/jovyan/Deepfake_Low Resource_Low Resolution/DFD-HR/run_dfd_hr_df40.py True
DFD-HR checkpoint: /home/jovyan/Deepfake_Low Resource_Low Resolution/DFD-HR/ckpt_best.pth True

Running:
/opt/conda/bin/python /home/jovyan/Deepfake_Low Resource_Low Resolution/DFD-HR/run_dfd_hr_df40.py --repo_root /home/jovyan/Deepfake_Low Resource_Low Resolution/DFD-HR --metadata_csv /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40_ood_fake_test_all_models/df40_fake_test_manifest_for_eval.csv --ckpt_path /home/jovyan/Deepfake_Low Resource_Low Resolution/DFD-HR/ckpt_best.pth --detector_yaml /home/jovyan/Deepfake_Low Resource_Low Resolution/DFD-HR/training/config/detector/dfd_hr.yaml --test_config /home/jovyan/Deepfake_Low Resource_Low Resolution/DFD-HR/training/config/test_config.yaml --out_dir /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40_ood_fake_test_all_models/outputs_dfd_hr_df40_indomain_test --split test --image_size 448 --frames_per_video 1 --batch_size

Sampling videos: 100%|██████████| 175667/175667 [28:46<00:00, 101.75it/s]


Rows: 175667
Videos/groups: 175667
col_0   count
label        
1      175667
Detector name: dfd_hr
Loaded detector from registry key: dfd_hr


Loading weights: 100%|██████████| 590/590 [00:00<00:00, 8101.72it/s]


Missing keys: 0
Unexpected keys: 0


Predicting:  63%|██████▎   | 110424/175667 [21:55:48<13:20:24,  1.36it/s]IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

Predicting: 100%|██████████| 175667/175667 [34:56:50<00:00,  1.40it/s]   



Metrics:
Level  Accuracy  AUPRC  AUCROC  EER (Equal Error Rate)  Precision (0)  Recall (0)  F1 (0)  Precision (1)  Recall (1)   F1 (1)  Uncertain Rate (%)
Frame    0.6873    NaN     NaN                     NaN            0.0         0.0     0.0            1.0      0.6873 0.814675           19.745314
Video    0.6873    NaN     NaN                     NaN            0.0         0.0     0.0            1.0      0.6873 0.814675           19.745314

Inference:
 frames_scored  videos_scored  total_seconds  time_per_frame_ms      fps device  batch_size  image_size
        175667         175667  125810.356504         716.186629 1.396284   cuda           1         448

Saved to: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40_ood_fake_test_all_models/outputs_dfd_hr_df40_indomain_test
Finished in 2127.06 minutes with return code 0


In [4]:
!pip install pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 116.2 kB/s eta 0:00:00a 0:00:01
  Using cached numpy-2.4.6-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 15.2 MB/s eta 0:00:00:00:01
Using cached numpy-2.4.6-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.9 MB)
